# 双支天线长度预测 MLP 训练笔记
本 Notebook 展示了如何从包含反射系数幅值和相位的 CSV 数据集中训练一个预测双支天线最佳长度 (l1, l2) 的多层感知机。
在载入数据时，会自动把相位拆解为 sin/cos 两个特征，并与幅值 L 一起形成 3 维训练输入，以增强模型的表达能力。
Notebook 仅依赖 Python 标准库，方便在依赖受限的环境中直接运行。



## 如何在 Jupyter Notebook 中运行训练
1. 打开本 Notebook 后，先执行下方 "导入所需模块" 的代码单元，以加载 `simple_mlp` 辅助模块。
2. 接着运行 "配置区域" 单元，根据需要调整 CSV 路径或训练超参数。
3. 如果你还没有准备好的数据，可以继续运行 "生成演示用 CSV" 单元，它会在 `data/` 目录下生成 2000 条训练样本。
4. 最重要的训练步骤在 "定义并训练模型" 单元，运行该单元即可启动训练过程。
5. 最后运行 "训练集与测试集表现评估" 单元，查看 MSE 以及前几条预测结果。

你也可以通过菜单 "运行 -> 运行全部" 一次性顺序执行所有单元，Notebook 会自动在缺失数据时生成演示数据并完成训练。


In [ ]:
# 导入所需模块（全部来自标准库或 simple_mlp 辅助模块）
from pathlib import Path
from typing import Sequence
import math
import random

from simple_mlp import (
    SimpleMLP,
    Standardizer,
    read_csv_dataset,
    mean_squared_error,
    format_predictions,
    plot_loss_curve,
)


In [ ]:
# ===== 配置区域 =====
project_dir = Path('.')
train_csv_path = project_dir / 'data' / 'train.csv'
test_csv_path = project_dir / 'data' / 'test.csv'

raw_feature_columns = ['reflection_magnitude_L', 'reflection_phase']
expanded_feature_names = [
    'reflection_magnitude_L',
    'reflection_phase_sin',
    'reflection_phase_cos',
]
target_columns = ['optimal_length_l1', 'optimal_length_l2']

NUM_EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
L1_COEFF = 1e-5
RANDOM_SEED = 42
random.seed(RANDOM_SEED)



In [ ]:
# ===== 可选：生成演示用 CSV（当指定文件不存在或列不匹配时） =====
def generate_demo_csv(path: Path, with_targets: bool = True, num_rows: int = 2000) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    rng = random.Random(RANDOM_SEED)
    rows = []
    for _ in range(num_rows):
        magnitude = rng.uniform(0.0, 1.0)
        phase = rng.uniform(-math.pi, math.pi)
        row = {
            'reflection_magnitude_L': f"{magnitude:.6f}",
            'reflection_phase': f"{phase:.6f}",
            'reflection_phase_sin': f"{math.sin(phase):.6f}",
            'reflection_phase_cos': f"{math.cos(phase):.6f}",
        }
        if with_targets:
            real_part = magnitude * math.cos(phase)
            imag_part = magnitude * math.sin(phase)
            base_length = 0.3 + 0.25 * magnitude
            delta = 0.15 * real_part - 0.1 * imag_part
            coupling = 0.05 * math.sin(2 * phase)
            noise = 0.03 * rng.gauss(0, 1)
            length_l1 = max(0.0, min(7.0, 7 * (base_length + delta + noise)))
            length_l2 = max(0.0, min(7.0, 7 * (base_length - delta + coupling + noise)))
            row['optimal_length_l1'] = f"{length_l1:.6f}"
            row['optimal_length_l2'] = f"{length_l2:.6f}"
        rows.append(row)

    feature_header = [
        'reflection_magnitude_L',
        'reflection_phase',
        'reflection_phase_sin',
        'reflection_phase_cos',
    ]
    header = list(feature_header)
    if with_targets:
        header.extend(target_columns)

    with path.open('w', encoding='utf-8', newline='') as handle:
        handle.write(','.join(header) + '\n')
        for row in rows:
            handle.write(','.join(row.get(column, '') for column in header) + '\n')
    print(f"Demo CSV generated at: {path} (rows={num_rows}, targets={'yes' if with_targets else 'no'})")

def csv_header_matches(path: Path, expected_headers: Sequence[Sequence[str]]) -> bool:
    if not path.exists():
        return False
    with path.open('r', encoding='utf-8') as handle:
        header_line = handle.readline().strip()
    if not header_line:
        return False
    existing = [column.strip() for column in header_line.split(',') if column.strip()]
    return any(existing == list(header) for header in expected_headers)

generated_feature_header = [
    'reflection_magnitude_L',
    'reflection_phase',
    'reflection_phase_sin',
    'reflection_phase_cos',
]
expected_train_headers = [
    raw_feature_columns + target_columns,
    expanded_feature_names + target_columns,
    generated_feature_header + target_columns,
]
expected_test_headers = [
    raw_feature_columns,
    expanded_feature_names,
    generated_feature_header,
    raw_feature_columns + target_columns,
    expanded_feature_names + target_columns,
    generated_feature_header + target_columns,
]

if not csv_header_matches(train_csv_path, expected_train_headers):
    generate_demo_csv(train_csv_path, with_targets=bool(target_columns), num_rows=2000)

if not csv_header_matches(test_csv_path, expected_test_headers):
    generate_demo_csv(test_csv_path, with_targets=bool(target_columns), num_rows=400)



### 常见数据报错说明
如果你之前使用过旧版 Notebook，它生成的 CSV 标头可能会把 `optimal_length_l1`、`optimal_length_l2` 误写成 `optimal_length_11`、`optimal_length_12`（字母 l 被数字 1 替换）。
本 Notebook 现在会自动检测并修正这类旧列名，以确保训练能够顺利读取目标值。

In [ ]:
# ===== 读取数据并进行标准化 =====
legacy_aliases = {
    'optimal_length_11': 'optimal_length_l1',
    'optimal_length_12': 'optimal_length_l2',
}

def rename_legacy_headers(path: Path, aliases) -> bool:
    # 将旧列名重命名为当前约定的列名，返回是否发生修改。
    if not path.exists():
        return False
    with path.open('r', encoding='utf-8') as handle:
        lines = handle.readlines()
    if not lines:
        return False
    header = [column.strip() for column in lines[0].strip().split(',') if column.strip()]
    replaced = False
    for idx, column in enumerate(header):
        if column in aliases:
            header[idx] = aliases[column]
            replaced = True
    if not replaced:
        return False
    lines[0] = ','.join(header) + '\n'
    with path.open('w', encoding='utf-8', newline='') as handle:
        handle.writelines(lines)
    alias_pairs = ', '.join(f"{old}→{new}" for old, new in aliases.items())
    print(f"检测到 {path.name} 使用了旧版列名，已自动替换为: {alias_pairs}。")
    return True

def load_dataset_with_legacy_support(path: Path, feature_columns: Sequence[str], target_columns: Sequence[str]):
    try:
        return read_csv_dataset(path, feature_columns, target_columns)
    except ValueError as exc:
        message = str(exc)
        if target_columns and 'Missing target columns' in message:
            if rename_legacy_headers(path, legacy_aliases):
                return read_csv_dataset(path, feature_columns, target_columns)
        raise

def expand_phase_features(feature_rows):
    magnitude_index = raw_feature_columns.index('reflection_magnitude_L')
    phase_index = raw_feature_columns.index('reflection_phase')
    expanded = []
    for row in feature_rows:
        magnitude = row[magnitude_index]
        phase = row[phase_index]
        expanded.append([magnitude, math.sin(phase), math.cos(phase)])
    return expanded

def load_features_with_phase_expansion(path: Path, target_cols: Sequence[str]):
    try:
        raw_features, targets = load_dataset_with_legacy_support(path, raw_feature_columns, target_cols)
    except ValueError as exc:
        message = str(exc)
        if 'Missing feature columns' in message:
            expanded_features, targets = load_dataset_with_legacy_support(path, expanded_feature_names, target_cols)
            return None, expanded_features, targets
        raise
    expanded_features = expand_phase_features(raw_features)
    return raw_features, expanded_features, targets

train_features_raw, train_features_expanded, train_targets = load_features_with_phase_expansion(train_csv_path, target_columns)
try:
    test_features_raw, test_features_expanded, test_targets = load_features_with_phase_expansion(test_csv_path, target_columns)
except ValueError as exc:
    print(f"Test CSV 缺少目标列: {exc}. 将仅基于输入特征进行预测。")
    test_features_raw, test_features_expanded, _ = load_features_with_phase_expansion(test_csv_path, [])
    test_targets = None

standardizer = Standardizer.fit(train_features_expanded)
train_features = standardizer.transform(train_features_expanded)
test_features = standardizer.transform(test_features_expanded)

print(f'Training samples: {len(train_features)}')
print(f'Test samples: {len(test_features)}')



In [ ]:
# ===== 定义并训练模型 =====
mlp = SimpleMLP(
    layer_sizes=[len(expanded_feature_names), 32, 16, len(target_columns)],
    learning_rate=LEARNING_RATE,
    l1_coeff=L1_COEFF,
    seed=RANDOM_SEED,
)
loss_history = mlp.train(
    features=train_features,
    targets=train_targets,
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose_interval=10,
)
print(f'最后一个 epoch 的平均批次损失: {loss_history[-1]:.6f}')



In [ ]:
# ===== 训练集与测试集表现评估 =====
train_predictions = mlp.predict_batch(train_features)
train_mse = mean_squared_error(train_predictions, train_targets)
print(f'Train MSE: {train_mse:.6f}')

test_predictions = mlp.predict_batch(test_features)
if test_targets is not None:
    test_mse = mean_squared_error(test_predictions, test_targets)
    print(f'Test MSE: {test_mse:.6f}')
else:
    print('Test CSV 未提供目标列，无法计算 MSE。')

print('测试集预测示例:')
print(
    format_predictions(
        test_features_expanded,
        test_predictions,
        test_targets,
        expanded_feature_names,
        target_columns,
    )
)



In [ ]:
# ===== 可视化损失曲线 =====
artifacts_dir = project_dir / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)
loss_curve_path = artifacts_dir / 'loss_curve.svg'
plot_loss_curve(loss_history, save_path=loss_curve_path)
print(f'损失曲线已保存至: {loss_curve_path}')
try:
    from IPython.display import SVG, display
except ImportError:
    SVG = None
    display = None
if SVG is not None and display is not None:
    display(SVG(filename=str(loss_curve_path)))
else:
    print('可以手动打开 SVG 文件查看曲线。')


## 小结
- 通过标准化反射系数输入特征并训练 `SimpleMLP`，可以在受限环境中完成双输出天线长度预测。
- 可以根据需要调整隐藏层规模、训练轮数以及正则化强度，以获得更好的效果。


In [ ]:
# ===== 测试集误差指标与预测结果导出 =====
import csv

if test_targets is None:
    print('Test CSV 未提供目标列，无法计算误差指标与导出预测结果。')
else:
    mse = mean_squared_error(test_predictions, test_targets)
    rmse = math.sqrt(mse)

    total_abs_error = 0.0
    value_count = 0
    for pred_row, true_row in zip(test_predictions, test_targets):
        for pred_value, true_value in zip(pred_row, true_row):
            total_abs_error += abs(pred_value - true_value)
            value_count += 1
    mae = total_abs_error / value_count if value_count else float('nan')

    true_values = [value for row in test_targets for value in row]
    pred_values = [value for row in test_predictions for value in row]
    if true_values:
        true_mean = sum(true_values) / len(true_values)
        ss_res = sum((true - pred) ** 2 for true, pred in zip(true_values, pred_values))
        ss_tot = sum((true - true_mean) ** 2 for true in true_values)
        r2 = 1.0 - ss_res / ss_tot if ss_tot else float('nan')
    else:
        r2 = float('nan')

    print(f'测试集均方误差 (MSE): {mse:.6f}')
    print(f'测试集均方根误差 (RMSE): {rmse:.6f}')
    print(f'测试集平均绝对误差 (MAE): {mae:.6f}')
    if math.isnan(r2):
        print('测试集决定系数 (R^2): 无法计算（测试集目标值方差为 0）')
    else:
        print(f'测试集决定系数 (R^2): {r2:.6f}')

    artifacts_dir.mkdir(parents=True, exist_ok=True)
    predictions_csv_path = artifacts_dir / 'test_predictions.csv'

    export_feature_header = []
    if test_features_raw is not None:
        export_feature_header.extend(raw_feature_columns)
        export_feature_header.extend(expanded_feature_names[1:])
    else:
        export_feature_header.extend(expanded_feature_names)

    header = (
        export_feature_header
        + [f'pred_{name}' for name in target_columns]
        + [f'true_{name}' for name in target_columns]
    )

    with predictions_csv_path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.writer(handle)
        writer.writerow(header)
        for idx, (pred_row, true_row) in enumerate(zip(test_predictions, test_targets)):
            row_values = []
            if test_features_raw is not None:
                row_values.extend(test_features_raw[idx])
                row_values.extend(test_features_expanded[idx][1:])
            else:
                row_values.extend(test_features_expanded[idx])
            row_values.extend(pred_row)
            row_values.extend(true_row)
            writer.writerow(row_values)

    print(f'预测结果已保存至: {predictions_csv_path}')

